# Solution 3 — Integration and sensitivity


In [ ]:
# Locate the repository so these editable modules work without installation.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Reload edited model modules before each cell during tutorial development.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np

from casadi_tutorial.integration import euler_step, rk4_step


def rhs(x, u):
    """Return the scalar dynamics x_dot = -u*x."""
    return -u * x


def terminal_function(step_method, step_size):
    """Build the terminal state and input-sensitivity Function."""
    u = ca.MX.sym("u")
    x = ca.MX(1)
    number_of_steps = round(1 / step_size)
    for _ in range(number_of_steps):
        x = step_method(rhs, x, u, step_size)
    return ca.Function(
        f"terminal_{step_method.__name__.split('_')[0]}_{number_of_steps}",
        [u],
        [x, ca.jacobian(x, u)],
        ["u"],
        ["x_terminal", "dx_terminal_du"],
    )


euler_terminal = terminal_function(euler_step, 0.1)
rk4_terminal = terminal_function(rk4_step, 0.1)
exact_terminal = np.exp(-2)
assert abs(float(rk4_terminal(2)[0]) - exact_terminal) < abs(
    float(euler_terminal(2)[0]) - exact_terminal
)
np.testing.assert_allclose(
    float(rk4_terminal(2)[1]),
    -exact_terminal,
    rtol=2e-4,
)

# Plot the full state response under the constant manipulated input u=2.
step_size = 0.1
number_of_steps = round(1 / step_size)
time = step_size * np.arange(number_of_steps + 1)
u_history = np.full(number_of_steps, 2.0)
x_euler = [1.0]
x_rk4 = [1.0]
for u_value in u_history:
    x_euler.append(float(euler_step(rhs, x_euler[-1], u_value, step_size)))
    x_rk4.append(float(rk4_step(rhs, x_rk4[-1], u_value, step_size)))
x_exact = np.exp(-2.0 * time)

figure, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
axes[0].plot(time, x_euler, label="Euler")
axes[0].plot(time, x_rk4, label="RK4")
axes[0].plot(time, x_exact, "--", label="exact")
axes[0].set_ylabel("state x")
axes[0].legend()
axes[0].grid(True)
axes[1].step(time[:-1], u_history, where="post")
axes[1].set(xlabel="time", ylabel="input u")
axes[1].grid(True)
figure.tight_layout()
plt.show()